# BazaarBot GRPO Training (Kaggle dual-T4)

End-to-end loop:
1. Auth into HF using Kaggle Secret `HF_TOKEN`.
2. Clone this project so we can import `bazaarbot_env`.
3. Load base model + tokenizer with 4-bit quant, attach LoRA adapters.
4. Define a reward function that runs `rollout_episode` per prompt.
5. Train with TRL's `GRPOTrainer`.
6. Push adapters to `PayMyBills/bestdealbot`.

**Before running:**
- Kaggle notebook settings → Accelerator: GPU T4 x2, Internet: On.
- Add-ons → Secrets → add `HF_TOKEN` (HF write token).
- Set `GITHUB_REPO` below to the pushed project URL.

**Quota:** one full run (~2–3 epochs over ~500 rollouts) fits inside Kaggle's 12-hour session.

## 1 · Install deps

In [ ]:
!pip install -q --upgrade "trl>=0.12" "peft>=0.13" "transformers>=4.46" "accelerate>=1.1" "bitsandbytes>=0.44" "datasets>=3.0" huggingface_hub

## 2 · HF auth (via Kaggle Secret)

In [ ]:
import os
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("HF_TOKEN")

os.environ["HF_TOKEN"] = secret_value_0
login(token=secret_value_0)
print("HF login OK")

## 3 · Pull the bazaarbot_env package

Two options — pick one:
- **A** (recommended): push the repo to GitHub and clone it here.
- **B**: upload the `bazaarbot_env/` folder as a Kaggle Dataset and mount it.


In [ ]:
# Option A — git clone.  Replace with your repo URL once pushed.
GITHUB_REPO = "https://github.com/paymybills/BazaarBATNA.git"
!git clone {GITHUB_REPO} /kaggle/working/metathon

import sys
sys.path.insert(0, "/kaggle/working/metathon")

from bazaarbot_env import (
    BazaarGymEnv, rollout_episode, format_observation,
    parse_action, DEFAULT_SYSTEM_PROMPT, TASKS,
)

print("bazaarbot_env loaded. Tasks:", list(TASKS.keys()))

## 4 · Base model + LoRA

**Qwen3.5-4B** is the sweet spot: newest Qwen (Mar 2026), 5B effective params, fits
on one T4 with 4-bit + LoRA, produces valid action JSON out-of-the-box so GRPO
learns strategy rather than syntax, and quantizes to ~2.5GB GGUF for local Ollama
on your 2050.

Drop to `Qwen/Qwen3.5-2B` if rollouts OOM or you want `num_generations=8` for
lower-variance GRPO advantages. Qwen3.5 models are VLMs (Image-Text-to-Text) —
the vision encoder is unused during negotiation training, wasted capacity we'll
exploit later when we wire in listing images.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

BASE_MODEL = "Qwen/Qwen3.5-4B"
REPO_ID    = "PayMyBills/bestdealbot"

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
)
model = prepare_model_for_kbit_training(model)

lora = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()

## 5 · Prompt dataset

For GRPO we need prompts; the model generates N completions per prompt and the
reward function scores each.  Here a **prompt = (task, seed)** pair.  The reward
function samples a full episode per (prompt, completion) using that task/seed,
using the current model as the policy.

In [ ]:
import random
from datasets import Dataset

# amazon_realistic samples a real product + MRP/street-price per episode
# from data/amazon.csv (1417 unique listings).  Mixing with single_deal
# keeps the toy distribution as a sanity anchor.
TRAIN_TASKS = ["amazon_realistic", "amazon_realistic", "single_deal"]
N_PROMPTS = 256

rng = random.Random(0)
train_rows = []
for i in range(N_PROMPTS):
    task = rng.choice(TRAIN_TASKS)
    seed = rng.randint(0, 1_000_000)
    env = BazaarGymEnv(task_name=task, seed=seed)
    obs, _ = env.reset()
    user_turn = format_observation(obs)
    chat = tokenizer.apply_chat_template(
        [
            {"role": "system", "content": DEFAULT_SYSTEM_PROMPT},
            {"role": "user",   "content": user_turn},
        ],
        tokenize=False,
        add_generation_prompt=True,
    )
    train_rows.append({"prompt": chat, "task": task, "seed": seed})

train_ds = Dataset.from_list(train_rows)
print("Training prompts:", len(train_ds))
print("Sample task mix:", dict((t, sum(1 for r in train_rows if r['task'] == t)) for t in set(TRAIN_TASKS)))

## 5.5 · SFT warmup (teach the model to emit valid JSON)

Cold 4-bit Qwen3.5 without any SFT can't reliably produce our action JSON format
in response to our prompt — it generates free-form text, every completion hits
a parse error, GRPO advantages collapse to zero, nothing trains.

Before GRPO, do a quick SFT pass: generate 256 (prompt → rule-based-buyer-action)
pairs using a heuristic buyer, fine-tune the LoRA for 1 epoch on those. ~3-5 min
on T4. After this, `_generate()` reliably outputs `{"action": ..., "price": ...}`
and GRPO can start doing real strategy learning.

In [ ]:
import json
import random
from datasets import Dataset


def _rule_based_buyer(obs: dict) -> dict:
    """Simple heuristic that produces reasonable opening actions.

    This is the target behavior for SFT warmup - not a clever policy,
    just 'always emit valid JSON, with price somewhere sensible'.
    """
    ask    = obs.get("seller_asking_price") or obs.get("opponent_last_offer") or 100
    budget = obs.get("own_private_budget") or 100
    rnd    = obs.get("current_round") or 0
    last   = obs.get("own_last_offer")

    # Accept if seller is already at <= 50% of our budget (cheap).
    if ask <= budget * 0.5:
        return {"action": "accept", "price": None}
    # Walk if seller demands more than budget.
    if ask > budget:
        return {"action": "walk", "price": None}
    # Otherwise offer: first round anchors at 30% of ask, then creeps up.
    if rnd == 0 or last is None:
        price = ask * random.uniform(0.25, 0.40)
    else:
        price = last + (ask - last) * random.uniform(0.2, 0.35)
    price = max(1.0, min(price, budget * 0.8))
    return {"action": "offer", "price": round(price, 2)}


def _build_sft_sample(task: str, seed: int) -> dict:
    """One (prompt, completion) pair for SFT.

    Completion is exactly the JSON the rule-based buyer would emit,
    so SFT teaches the output format without baking in a bad policy.
    GRPO will overwrite the strategy later; this just establishes syntax.
    """
    env = BazaarGymEnv(task_name=task, seed=seed)
    obs, _ = env.reset()
    action = _rule_based_buyer(obs)
    user_turn = format_observation(obs)

    # Chat template with the assistant turn filled in (no generation prompt).
    full = tokenizer.apply_chat_template(
        [
            {"role": "system",    "content": DEFAULT_SYSTEM_PROMPT},
            {"role": "user",      "content": user_turn},
            {"role": "assistant", "content": json.dumps(action)},
        ],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": full}


SFT_N = 256
sft_rng = random.Random(1)
sft_rows = []
for _ in range(SFT_N):
    task = sft_rng.choice(["amazon_realistic", "single_deal", "asymmetric_pressure"])
    seed = sft_rng.randint(0, 1_000_000)
    sft_rows.append(_build_sft_sample(task, seed))

sft_ds = Dataset.from_list(sft_rows)
print(f"SFT dataset: {len(sft_ds)} examples")
print("Sample:")
print(sft_ds[0]["text"][-300:])

In [ ]:
from trl import SFTConfig, SFTTrainer

sft_cfg = SFTConfig(
    output_dir="/kaggle/working/bestdealbot-sft",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=1e-4,
    num_train_epochs=1,
    logging_steps=5,
    save_strategy="no",             # no mid-run checkpoints, we'll save at end
    bf16=True,
    report_to="none",
    max_length=1024,
    dataset_text_field="text",
    packing=False,
)

sft_trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    args=sft_cfg,
    train_dataset=sft_ds,
)
sft_trainer.train()
print("SFT warmup complete.")

# Sanity check: does the model now emit valid JSON?
env = BazaarGymEnv(task_name="amazon_realistic", seed=7)
obs, _ = env.reset()
chat = tokenizer.apply_chat_template(
    [{"role": "system", "content": DEFAULT_SYSTEM_PROMPT},
     {"role": "user",   "content": format_observation(obs)}],
    tokenize=False, add_generation_prompt=True,
)
for i in range(3):
    out = _generate(chat, max_new_tokens=48) if "_generate" in dir() else None
    if out is None:
        # _generate not defined yet; inline a quick version
        import torch
        with torch.no_grad():
            inputs = tokenizer(chat, return_tensors="pt", truncation=True, max_length=1024).to(model.device)
            g = model.generate(**inputs, max_new_tokens=48, do_sample=True, temperature=0.7,
                               top_p=0.9, pad_token_id=tokenizer.pad_token_id)
            out = tokenizer.decode(g[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    print(f"sample {i}: {out!r}")

## 6 · Reward function (full-episode rollout)

The reward for a single generated completion is the **terminal graded score of the full episode** played starting from that completion as the first action.  We run the rest of the episode with the same model.

This is expensive — N_rollouts × ~5–8 continuation turns of forward passes per GRPO step — but correct: the GRPO advantage corresponds to "how good was the negotiation you produced" rather than just "did you output valid JSON".

**Speed knobs** (if rollouts are too slow on T4):
- Lower `max_new_tokens` in `_generate` (64 → 32).
- Lower `MAX_CONTINUATION_TURNS` (12 → 6). Most single_deal episodes close in 4–5 turns anyway.
- Skip the continuation loop entirely and score only the first action (set `MAX_CONTINUATION_TURNS = 0`). Reward becomes "did this first offer produce a reasonable opening" — weaker signal but ~10× faster.

In [ ]:
# Speed knobs.  On T4 with Qwen3.5's hybrid linear-attention, rollouts are
# the dominant cost; cap them tightly so GRPO steps actually complete.
#
# FAST_MODE=True uses a shaped first-step reward instead of continuation
# rollouts.  Shaping gives enough dynamic range that sampled completions
# produce distinct rewards => nonzero GRPO advantages => actual gradient.
FAST_MODE               = True
MAX_CONTINUATION_TURNS  = 0 if FAST_MODE else 6
GEN_MAX_NEW_TOKENS      = 48

@torch.no_grad()
def _generate(prompt_text: str, max_new_tokens: int = GEN_MAX_NEW_TOKENS) -> str:
    inputs = tokenizer(prompt_text, return_tensors="pt", truncation=True, max_length=1024).to(model.device)
    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        pad_token_id=tokenizer.pad_token_id,
    )
    gen_ids = out[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(gen_ids, skip_special_tokens=True)


def _shaped_first_step_reward(obs: dict, action: dict, step_reward: float) -> float:
    """Reward for FAST_MODE: env step reward + opening-offer shaping.

    Returns a value roughly in [-0.5, 1.2] with meaningful variance across
    different policy samples, so GRPO's group-normalized advantages are
    nonzero even when all completions parse successfully.
    """
    if action.get("_parse_error"):
        return -0.3
    act_type = action.get("action")
    ask = obs.get("seller_asking_price") or 0
    budget = obs.get("own_private_budget") or 0
    price = action.get("price") or 0

    if act_type == "accept":
        # Accepting the very first seller ask is almost always bad
        return -0.2
    if act_type == "walk":
        return -0.3
    if ask <= 0 or budget <= 0:
        return float(step_reward)
    if price <= 0 or price > budget:
        return -0.3

    # Opening offer shaping: reward lower offers, but penalize absurdly low
    # offers that will cause a seller walk.
    ratio = price / ask
    if ratio < 0.15:
        shape = 0.1                # too aggressive
    elif ratio < 0.35:
        shape = 1.0                # sweet spot: strong anchor
    elif ratio < 0.55:
        shape = 0.6                # decent
    elif ratio < 0.75:
        shape = 0.3                # weak anchor
    else:
        shape = 0.0                # basically capitulating
    return float(step_reward) + shape


def reward_fn(completions, prompts=None, completion_ids=None, **kwargs):
    """GRPO reward: completion becomes the first action; optionally roll out."""
    rewards = []
    tasks = kwargs.get("task") or ["amazon_realistic"] * len(completions)
    seeds = kwargs.get("seed") or [None] * len(completions)

    for completion, task, seed in zip(completions, tasks, seeds):
        env = BazaarGymEnv(task_name=task, seed=seed)
        obs, _ = env.reset()
        first_action = parse_action(
            completion,
            fallback_price=obs.get("own_private_budget", 100) * 0.3,
        )

        _obs, r, done, info = env.step(first_action)

        if MAX_CONTINUATION_TURNS > 0:
            history = []
            turns_left = MAX_CONTINUATION_TURNS
            while not done and turns_left > 0:
                user = format_observation(_obs, history=history)
                chat = tokenizer.apply_chat_template(
                    [
                        {"role": "system", "content": DEFAULT_SYSTEM_PROMPT},
                        {"role": "user",   "content": user},
                    ],
                    tokenize=False,
                    add_generation_prompt=True,
                )
                raw = _generate(chat)
                act = parse_action(raw, fallback_price=_obs.get("own_private_budget", 100) * 0.3)
                _obs, r, done, info = env.step(act)
                turns_left -= 1
            reward = float(env.score())
        else:
            reward = _shaped_first_step_reward(obs, first_action, r)

        rewards.append(reward)
    return rewards

## 7 · GRPO training

In [ ]:
from trl import GRPOConfig, GRPOTrainer

# Checkpoint to HF on every save_steps so a Kaggle timeout doesn't lose the run.
# Requires `hub_model_id` repo to exist and be writable with the logged-in token.
grpo_cfg = GRPOConfig(
    output_dir="/kaggle/working/bestdealbot-ckpt",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_generations=2,
    max_completion_length=48,
    learning_rate=5e-6,
    num_train_epochs=1,
    max_steps=30,                    # prove the loop, scale later
    logging_steps=1,
    save_steps=10,
    save_total_limit=2,              # rolling window of last 2 checkpoints
    bf16=True,
    report_to="none",
    remove_unused_columns=False,
    # push to HF on every save
    push_to_hub=True,
    hub_model_id="PayMyBills/bestdealbot",
    hub_strategy="checkpoint",
    hub_private_repo=False,
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=reward_fn,
    args=grpo_cfg,
    train_dataset=train_ds,
)

trainer.train()

## 8 · Push adapters to HF

Only the LoRA adapters get pushed (~50MB), not the merged model.  Merge locally
after download to avoid Kaggle network bottleneck.

In [ ]:
model.save_pretrained("/kaggle/working/bestdealbot-adapter")
tokenizer.save_pretrained("/kaggle/working/bestdealbot-adapter")

from huggingface_hub import HfApi
api = HfApi()
api.upload_folder(
    folder_path="/kaggle/working/bestdealbot-adapter",
    repo_id=REPO_ID,
    repo_type="model",
    commit_message="GRPO adapter from Kaggle run",
)
print(f"Pushed to https://huggingface.co/{REPO_ID}")

## Local post-training

```bash
# pull adapter
hf download PayMyBills/bestdealbot --local-dir ~/models/bdb

# merge base + adapter (run on your laptop)
python -m peft merge_and_unload \
    --base-model Qwen/Qwen3.5-4B \
    --adapter  ~/models/bdb \
    --out      ~/models/bdb-merged

# convert to GGUF
python llama.cpp/convert_hf_to_gguf.py ~/models/bdb-merged \
    --outfile bdb.gguf --outtype q4_k_m

# register with Ollama
cat > Modelfile <<'EOF'
FROM ./bdb.gguf
SYSTEM """You are a skilled buyer negotiating at an Indian bazaar."""
EOF
ollama create bestdealbot -f Modelfile
ollama run bestdealbot
```